In [ ]:
# TODO:
# final visualization of centroids (hard bc 40 dimensions...)?? 

# clean up code (image saving, etc)?
# re-run final, clean code?

## Setup

IMPORTANT: in case files in /home/ubuntu/Project/libero_development/src/ are changed, must also submit them to workers (to avoid reading from an old file on WORKER'S disk): from head VM's terminal, run:

```bash
SRC_DIR="/home/ubuntu/Project/libero_development/src"
DEST_DIR="/home/ubuntu/Project/libero_development/src"

WORKER_IPS=(
    "10.67.22.254" "10.67.22.34" "10.67.22.145" "10.67.22.121"
    "10.67.22.192" "10.67.22.18" "10.67.22.187" "10.67.22.48"
)

for ip in "${WORKER_IPS[@]}"; do
    echo "Syncing src/ to $ip..."
    ssh -o StrictHostKeyChecking=no ubuntu@"$ip" "mkdir -p $DEST_DIR"
    rsync -avz --exclude '__pycache__/' "$SRC_DIR/" ubuntu@"$ip":"$DEST_DIR/"
done
```


IMPORTANT: to kill existing Dask processes (from bash):
```bash
for ip in 10.67.22.194 10.67.22.254 10.67.22.34 10.67.22.145 10.67.22.121 10.67.22.192 10.67.22.18 10.67.22.187 10.67.22.48; do
    echo "Killing on $ip..."
    ssh -o StrictHostKeyChecking=no ubuntu@"$ip" "pkill -9 -f dask-scheduler; pkill -9 -f dask-worker; pkill -9 -f dask_ssh" 
done
```

USEFUL: to combine the results of various csv files into a single csv:
```bash
head -n 1 file.csv > combined.csv
for file in *your_files*.csv; do tail -n +2 "$file" >> combined.csv; done
```

USEFUL: to check if a certain port occupied prevents strating worker(s):
```bash
ssh ubuntu@10.67.22.194 "lsof -i :8786"
```
then to kill eg this PID:
```bash
ssh ubuntu@10.67.22.194 "fuser -k 8786/tcp"
```


In [ ]:
from src.launch_cluster import launch_cluster, shutdown_cluster

In [ ]:
import numpy as np
import pandas as pd
from src.kmeans_parallel import kmeans_parallel
from src.data_loader import load_dataset
from src.benchmark import run_single_test, run_benchmark, calculate_inertia, combinations_fn

import time

In [ ]:
# --- Cluster ---
N_WORKERS = 8      # between 1 and 8 (nodes available in launch_cluster.py)
NUM_PARTITIONS = 8 * N_WORKERS   # empirical rule: >= n_threads_per_worker * n_workers
# --- k-means|| algorithm ---
#K = 500                # number of final clusters
#L = 250                # oversampling factor (absolute). Alternative: L = round(L_OVER_K * K)
#R = 10                 # number of rounds of the parallel initialization
#MAX_ITER_FIT = 100     # max iterations of the Lloyd's phase (fit)

SEED = 42

In [ ]:
# --- Dataset ---
DATASET_URL_10PC = "https://ndownloader.figshare.com/files/5976042"
# link used by sklearn function fetch_kddcup99 (original link gives 403 error)
#***for 10% dataset***

DATASET_URL_FULL="https://ndownloader.figshare.com/files/5976045"
#***FULL DATASET***

RAW_GZ_PATH_10PC   = "/home/ubuntu/Project/libero_development/data/kddcup_data.gz" # compressed (.gz) dataset file
PARQUET_PATH_10PC = '/tmp/kddcup_data_shards' # directory of Parquet shard files on master (one per partition)

RAW_GZ_PATH_FULL = "/home/ubuntu/backup/libero_development/data/kddcup_data_full.gz"
PARQUET_PATH_FULL = '/tmp/kddcup_data_full_shards'

# column names of KDD dataset, from source code of the above sklearn function;
# "protocol_type","service","flag" are non-numeric so they will be dropped later,
# as will be "label" and the constant column 'num_outbound_cmds' 
# (10% dataset might have more constant columns such as 'is_host_login')
COL_NAMES = [
    "duration","protocol_type","service","flag","src_bytes",
    "dst_bytes","land","wrong_fragment","urgent","hot",
    "num_failed_logins","logged_in","num_compromised","root_shell",
    "su_attempted","num_root","num_file_creations","num_shells",
    "num_access_files","num_outbound_cmds","is_host_login",
    "is_guest_login","count","srv_count","serror_rate",
    "srv_serror_rate","rerror_rate","srv_rerror_rate","same_srv_rate",
    "diff_srv_rate","srv_diff_host_rate","dst_host_count",
    "dst_host_srv_count","dst_host_same_srv_rate",
    "dst_host_diff_srv_rate","dst_host_same_src_port_rate",
    "dst_host_srv_diff_host_rate","dst_host_serror_rate",
    "dst_host_srv_serror_rate","dst_host_rerror_rate",
    "dst_host_srv_rerror_rate","label"
]

In [ ]:
# TODO: remove "/backup/" in cell above? (but it gets messy)

In [ ]:
# DO NOT RUN if already started! :
cluster, client = launch_cluster(N_WORKERS)

#### IF INSTEAD THE CLUSTER HAS ALREADY BEEN STARTED:

In [ ]:
from dask.distributed import Client

SCHEDULER_ADDRESS = "tcp://10.67.22.194:8786"

try:
    client = Client(SCHEDULER_ADDRESS, timeout="10s")
    print("Connected to cluster successfully!")
    print(f"Dask Dashboard link: {client.dashboard_link}")

except Exception as e:
    print(f"Connection error: {e}")

## Load dataset

In [ ]:
#from src.data_loader import load_dataset

#DATASET_URL_10PC = "https://ndownloader.figshare.com/files/5976042"
#RAW = "/home/ubuntu/Project/libero_development/data/kddcup_data.gz"   # già in cache, niente download
#PQ  = "/tmp/kddcup_data.parquet"

In [ ]:
#10 percent:

start=time.time()
X_bag_10_percent, (mean_ar, std_ar)=load_dataset(n_partitions=NUM_PARTITIONS,
                   client=client,
                   dataset_url=DATASET_URL_10PC,
                   raw_gz_path=RAW_GZ_PATH_10PC,
                   parquet_path=PARQUET_PATH_10PC,
                   col_names=COL_NAMES,
                   force_download=False)
end=time.time()
elapsed=end-start
print(f"Time elapsed: {elapsed:.2f} s")

#### FULL

In [ ]:
# full:

start=time.time()
X_bag_full, (mean_ar, std_ar)=load_dataset(n_partitions=NUM_PARTITIONS,
                   client=client,
                   dataset_url=DATASET_URL_FULL,
                   raw_gz_path=RAW_GZ_PATH_FULL,
                   parquet_path=PARQUET_PATH_FULL,
                   col_names=COL_NAMES,
                   force_download=True)
end=time.time()
elapsed=end-start
print(f"Time elapsed: {elapsed:.2f} s")

## Run experiments

#### Varying number of L/K

##### k=500

In [ ]:
combos = [                      
    #(N_WORKERS, 32, l_over_k, R),   # under-partitioned
    #(N_WORKERS, 64, l_over_k, R),   # balanced (1 part/thread)
    #(N_WORKERS, 128, 0.5, 5),
    (N_WORKERS, 128, 1, 5),   
    #(N_WORKERS, 128, 2, 5),  
    #(N_WORKERS, 128, 10, 5),
    #(N_WORKERS, 128, 0.1, 5), # done
]
K_VALUES=[500]
#K_VALUES=[500, 1000]

MAX_ITER_FIT=80 # for best convergence (conv criterion??)

avg_iters=10

In [ ]:
dataset_bag = X_bag_full
df = run_benchmark(client, X_bag=dataset_bag, combinations=combos,
                   k_values=K_VALUES, label="l_su_k_1_fixed_new_code",
                   max_iter_fit=MAX_ITER_FIT, seed=SEED,
                   averaging_iterations=avg_iters,
                   policy='fixed')

##### k=1000

In [ ]:
combos = [
    # num workers, num partitions, l over k, r
    #(N_WORKERS, 128, 1, 5), # fatto
    #(N_WORKERS, 128, 2, 5),  
    (N_WORKERS, 128, 10, 5), # fatto
    #(N_WORKERS, 128, 0.5, 5),
    #(N_WORKERS, 128, 0.1, 5),
]

K_VALUES=[1000]

MAX_ITER_FIT=80

avg_iters=5

In [ ]:
dataset_bag = X_bag_full
df = run_benchmark(client, X_bag=dataset_bag, combinations=combos,
                   k_values=K_VALUES, label="l_over_k_full_k1000_lok10",
                   max_iter_fit=MAX_ITER_FIT, seed=SEED, averaging_iterations=avg_iters)

In [ ]:
# note: less lloyd iteration (stopped bc below relative tolerance threshold - look at CSV) is ok because huge oversampling factor!

### Run with random (tabs 3,4)

In [ ]:
# idea: standard kmeans won't converge at correct cost ever
# so in order to show this, let it run for longer time than parallel kmeans 
# - that means longer than 2 minutes on average, so a max_iter_fit of
# about ***60*** to be sure (??)

In [ ]:
combos = [
    (N_WORKERS, 128, 0, 0)
    # R=0 per avere inizializzazione random (l can have any value, as it is not used)
]
avg_iters = 10
K_VALUES = [500, 1000]
MAX_ITER_FIT = 20 # as in the paper

In [ ]:
dataset_bag = X_bag_full
df = run_benchmark(client, X_bag=dataset_bag, combinations=combos,
                   k_values=K_VALUES, label="random_l_su_k",
                   max_iter_fit=MAX_ITER_FIT, seed=SEED, averaging_iterations=avg_iters,policy='fixed')

###### [OLD] Tables 3 and 4 (some things already done above, runtable34 does not save after each run)

In [ ]:
from src.paper_experiments import run_table34, table34_cost_table, table34_time_table

In [ ]:
df = run_table34(client=client, X_bag_full=X_bag_full, l_over_k_values=(1,2),include_random=True, n_runs=5)

In [ ]:
# debugging, skip:

In [ ]:
# check which function definition is seen by workers:

In [ ]:
def check_lloyd_pass_source():
    import src.kmeans_parallel as kmpar_source
    import inspect
    return inspect.getsource(kmpar_source._lloyd_pass)

In [ ]:
result = client.run(check_lloyd_pass_source)
# print only one worker's version, nicely formatted
first_worker = list(result.keys())[0]
print(result[first_worker])

In [ ]:
#def check_file_location():
#    import src.kmeans_parallel as m
#    return m.__file__

#print(client.run(check_file_location))

In [ ]:
# more debugging, to see if new code is mathematically correct (ie, compare single run to sklearn):

In [ ]:
import numpy as np
from sklearn.cluster import KMeans
from src.benchmark import run_single_test  # adjust import if needed

# Pull a manageable subsample (already a numpy array, or convert)
sample_size = 50000
k_test = 50

# If X_bag_full is a Dask array of rows, take sample and compute
X_sample = X_bag_full[:sample_size].compute()  # (sample_size, n_features)

# Your implementation via run_single_test
result, _ = run_single_test(
    client,                     # existing Dask client
    k=k_test,
    l=50,                      # oversampling factor
    r=5,                        # number of seeding rounds
    num_partitions=8,           # small for debugging
    max_iter_fit=100,           # enough iterations
    seed=42,
    X=X_sample,                 # pass the numpy array directly
    track_convergence=True,    
    track_centroids=True
)

# The result dict contains 'final_cost' (and 'initial_cost')
your_cost = result['final_cost']

# Sklearn reference
skl = KMeans(n_clusters=k_test, n_init=5, random_state=42).fit(X_sample)
sklearn_cost = skl.inertia_

print(f"Your cost: {your_cost:.2f}")
print(f"sklearn cost: {sklearn_cost:.2f}")
print(f"Ratio: {your_cost / sklearn_cost:.3f}")

In [ ]:
from sklearn.decomposition import PCA
import matplotlib.pyplot as plt

In [ ]:
# Extract histories
cost_history = result['cost_history']        # list of inertia per Lloyd iteration
iter_times = result['iter_times']            # list of times per iteration
n_centroids_history = result['n_centroids_history']  # list of candidate count after each seeding round

# 1. Plot cost vs iteration
plt.figure(figsize=(12, 4))
plt.subplot(1, 3, 1)
plt.plot(range(1, len(cost_history)+1), cost_history, marker='o')
plt.xlabel('Lloyd iteration')
plt.ylabel('Inertia')
plt.title('Cost convergence')
plt.grid(True, alpha=0.3)

# 2. Plot number of centroids candidates vs seeding round
plt.subplot(1, 3, 2)
plt.plot(range(1, len(n_centroids_history)+1), n_centroids_history, marker='s', color='orange')
plt.xlabel('Seeding round (1 to R)')
plt.ylabel('Number of candidates')
plt.title('Centroid pool growth')
plt.grid(True, alpha=0.3)


In [ ]:
#print(result)

In [ ]:
from sklearn.cluster import KMeans

# Import the needed builder (it's in benchmark.py)
from src.benchmark import _build_bag
from src.kmeans_parallel import kmeans_parallel

# --- 1. Prepare data ---
sample_size = 50000
k_test = 50
X_sample = X_bag_full[:sample_size].compute()  # (sample_size, n_features)

# Build the dask.array properly (scattered to workers) 
bag = _build_bag(client, X_sample, num_partitions=8)

# --- 2. Run your implementation (keep the clf object) ---
clf = kmeans_parallel(k=k_test, l=25, r=5)
clf.compute_starting_centroids(bag, seed=42, track_centroids=True, policy='fixed')
clf.fit(bag, max_iter=100, track_convergence=True)
your_cost = clf.inertia(bag)

# --- 3. sklearn reference ---
skl = KMeans(n_clusters=k_test, n_init=5, random_state=42).fit(X_sample)
sklearn_cost = skl.inertia_

print(f"Your cost: {your_cost:.2f}")
print(f"sklearn cost: {sklearn_cost:.2f}")
print(f"Ratio: {your_cost / sklearn_cost:.3f}")

In [ ]:
print(f"sklearn iterations for its BEST run: {skl.n_iter_}") 

In [ ]:
# --- 4. Plots ---
# Extract histories (available because we set track_*)
#cost_history = clf.cost_history_      # list per Lloyd iteration
#n_centroids_history = clf.n_centroids_history_  # list per seeding round
#final_centroids = clf.final_centroids

In [ ]:
# (c) 2D projection: data + final centroids
#pca = PCA(n_components=2)
#X_2d = pca.fit_transform(X_sample)
#centroids_2d = pca.transform(final_centroids)

# Subsample for visibility (plot 2000 random points)
#np.random.seed(42)
#idx = np.random.choice(len(X_2d), size=min(2000, len(X_2d)), replace=False)
#X_sub = X_2d[idx]

#plt.subplot()#(1, 3, 3)
#plt.scatter(X_sub[:, 0], X_sub[:, 1], s=2, alpha=0.5, label='Data points')
#plt.scatter(centroids_2d[:, 0], centroids_2d[:, 1], c='red', marker='X', s=100, label='Final centroids')
#plt.xlabel('PC1')
#plt.ylabel('PC2')
#plt.title('Data and centroids (PCA projection)')
#plt.legend()
#plt.grid(True, alpha=0.3)

#plt.tight_layout()
#plt.savefig('kmeans_debug_plots.png', dpi=150)
#plt.show()

## Number of centroids (table 5)

In [ ]:
# same code as that for l over k, BUT there it was run without track_centroids=True (now default):
# either skip this entirely or rerun some parts (some already done)
# ?

## Average number of LLoyd iterations (similar to table 6)

In [ ]:
# similar to table 6: can simply report number of lloyd iterations of kmeans parallel vs random 

## Run con diverso numero workers

In [ ]:
# as best combination (for cost) in figure 5.1 in original paper:
combos = [
    #(7, 128, 1, 5), # fatto
    (6, 128, 1, 5),
    (5, 128, 1, 5),
    (4, 128, 1, 5),
    (3, 128, 1, 5),
    (2, 128, 1, 5),
]

n_workers=[6,5,4,3,2]
avg_iters= 1

K_VALUES=[500]

MAX_ITER_FIT=80 # unimportant to get perfect convergence,
                # as we're investigating only ideal number of workers (wrt to computation time)
                # ok, but must be greater than 10. However, with 10% it's quick. Mattia

In [ ]:
shutdown_cluster(cluster, client)


In [ ]:
# 2 and 3 workers:

In [ ]:
# combination grid

num_part = [32, 64, 128, 256]
#num_works = [7, 6, 5, 4, 3, 2] # already done with 8 workers in partitions/
#num_part = [512, 1024, 2048]
num_works = [3, 2] # already done with 8 workers in partitions/
combos = [(w, p, 1, 5) for w in num_works for p in num_part ]
K_VALUES = [500,1000]
avg_iters = 10
MAX_ITER_FIT = 80

In [ ]:
frames = []
#shutdown_cluster(cluster, client)
j= 0
for combo in combos:
    #dask.config.set({"distributed.scheduler.work-stealing": True})
    n = combo[0]
    cluster, client = launch_cluster(n)
    mt=client.run(lambda: __import__('ctypes').CDLL("libc.so.6").malloc_trim(0))
    print(mt)

    combo = [combos[j],]
    j += 1
    
    X_bag_full, _ = load_dataset(n_partitions=8*n,
                   client=client,
                   dataset_url=DATASET_URL_FULL,
                   raw_gz_path=RAW_GZ_PATH_FULL,
                   parquet_path=PARQUET_PATH_FULL,
                   col_names=COL_NAMES,
                   force_download=False)  # rebuild bag on new client
    
    df_n = run_benchmark(client, X_bag=X_bag_full,
                         combinations=combo,
                         # function assigning appropriate values (in particular n partitions = 8 * n workers)
                         k_values=K_VALUES, label=f"{n}_workers_full_libero", 
                         max_iter_fit=MAX_ITER_FIT, seed=SEED,
                         averaging_iterations=avg_iters)
    
    df_n["num_workers"] = n
    frames.append(df_n)
    shutdown_cluster(cluster, client)

df_all = pd.concat(frames, ignore_index=True)

In [ ]:
frames = []
#shutdown_cluster(cluster, client)
j= 0
for n in n_workers:
    #dask.config.set({"distributed.scheduler.work-stealing": True})
    cluster, client = launch_cluster(n)
    mt=client.run(lambda: __import__('ctypes').CDLL("libc.so.6").malloc_trim(0))
    print(mt)

    combo = [combos[j],]
    j += 1
    
    X_bag_full, _ = load_dataset(n_partitions=8*n,
                   client=client,
                   dataset_url=DATASET_URL_FULL,
                   raw_gz_path=RAW_GZ_PATH_FULL,
                   parquet_path=PARQUET_PATH_FULL,
                   col_names=COL_NAMES,
                   force_download=False)  # rebuild bag on new client
    
    df_n = run_benchmark(client, X_bag=X_bag_full,
                         combinations=combo,
                         # function assigning appropriate values (in particular n partitions = 8 * n workers)
                         k_values=K_VALUES, label=f"{n}_workers_full_mattia", 
                         max_iter_fit=MAX_ITER_FIT, seed=SEED,
                         averaging_iterations=avg_iters)
    
    df_n["num_workers"] = n
    frames.append(df_n)
    shutdown_cluster(cluster, client)

df_all = pd.concat(frames, ignore_index=True)

### Figure 5.1

In [ ]:
# check Luca??

## Cluster shutdown

In [ ]:
shutdown_cluster(cluster, client)

## Compress `results` folder

In [ ]:
!tar -czf ../results.tar.gz -C .. results

In [ ]:
# can then download easily from inside Jupyter to one's local terminal